# GPU serving benchmark (vLLM on a free Colab T4)
**NOT RUN IN THIS ENVIRONMENT** (the repo's container has no GPU). Written to be run by you.

- Runtime: ~25-35 min on a free T4. **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**.
- What it does: replays the same Bitext-derived trace as the CPU baseline against vLLM (Qwen2.5-0.5B/1.5B-Instruct, fp16),
  profiles a short generation with `torch.profiler`, measures T4 bandwidth/GEMM as a second hardware profile for the perf model,
  and computes cost per 1k requests (CPU vs GPU).
- Output: `results/gpu_benchmark.json` and `results/hw_profile_t4.json` are downloaded at the end. Commit them into `results/`.
- Edit `REPO` below to your fork/clone URL if needed.

In [ ]:
REPO = "https://github.com/karthik-vi/llm-inference-gateway"
BRANCH = "claude/stoic-goldberg-2u4dpb"
!nvidia-smi
!pip -q install vllm httpx datasets pandas matplotlib
!git clone -q -b $BRANCH $REPO repo && cd repo && pip -q install -e . 

In [ ]:
import os, subprocess, time, json, urllib.request
os.chdir("/content/repo"); os.makedirs("data", exist_ok=True); os.makedirs("results", exist_ok=True)
# Bitext CSV (same file as scripts/download.sh) and the same trace builder (seed 0)
urllib.request.urlretrieve("https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset/resolve/main/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv", "data/bitext.csv")
from workloads.trace import build_trace, save
save(build_trace("data/bitext.csv", 400), "data/trace.jsonl")

In [ ]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # change to Qwen/Qwen2.5-1.5B-Instruct for the second run
srv = subprocess.Popen(["python", "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL, "--dtype", "float16",
    "--max-model-len", "2048", "--enable-prefix-caching", "--port", "8000"], stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)
import httpx
for _ in range(180):
    try:
        if httpx.get("http://127.0.0.1:8000/health").status_code == 200: break
    except Exception: pass
    time.sleep(2)
print("vLLM up")

In [ ]:
# Replay: same Poisson load generator as the CPU baseline, same SLO (p95 TTFT <= 2.0 s), higher rates (a T4 is faster).
import asyncio
from workloads.loadgen import run, summarize
from workloads.trace import load
sessions = load("data/trace.jsonl")
results = {}
for rate in [1, 2, 4, 8, 16, 32]:
    recs = asyncio.run(run("http://127.0.0.1:8000", sessions, rate, 60, 2.0, 30, seed=1, model=MODEL))
    results[rate] = summarize(recs, 60, 2.0); print(rate, {k: round(v, 3) for k, v in results[rate].items() if "ttft" in k or "goodput" in k})

In [ ]:
# torch.profiler on a short generation (HF transformers, separate from the server; free GPU memory first)
srv.terminate(); time.sleep(5)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL); m = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16).cuda()
x = tok("How do I cancel my order?", return_tensors="pt").to("cuda")
from torch.profiler import profile, ProfilerActivity
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
    m.generate(**x, max_new_tokens=32)
top = prof.key_averages().table(sort_by="cuda_time_total", row_limit=10); print(top)
prof_text = top

In [ ]:
# Second hardware profile for perfmodel (same keys as results/hw_profile_cpu.json; cores=1 => per-device)
import torch, time
n = 256 * 1024 * 1024 // 4
a = torch.randn(n, device="cuda"); b = torch.randn(n, device="cuda"); c = torch.empty_like(a)
torch.cuda.synchronize(); t = time.perf_counter()
for _ in range(20): torch.add(a, b, out=c)
torch.cuda.synchronize(); bw = 20 * 3 * n * 4 / (time.perf_counter() - t) / 1e9
x = torch.randn(4096, 4096, device="cuda", dtype=torch.float16); torch.cuda.synchronize(); t = time.perf_counter()
for _ in range(20): x @ x
torch.cuda.synchronize(); gf = 20 * 2 * 4096**3 / (time.perf_counter() - t) / 1e9
hw = {"cpu": "NVIDIA T4 (Colab)", "cores": 1, "triad_GBps_numpy_1thread": round(bw, 1), "gemm_fp32_GFLOPs_blas_all_threads": round(gf, 1)}
json.dump(hw, open("results/hw_profile_t4.json", "w"), indent=1); print(hw)

In [ ]:
# Cost per 1k requests. ASSUMPTIONS (edit!): T4 on-demand price and the CPU instance price you compare against.
GPU_USD_PER_HR, CPU_USD_PER_HR = 0.35, 0.17   # ASSUMED list prices, not measured here
best = max((r for r, s in results.items() if s["ttft_p95"] <= 2.0), default=None)
gpu_goodput = results[best]["goodput_rps"] if best else 0.0
out = {"model": MODEL, "gpu": "T4", "slo_ttft_p95_s": 2.0, "sweep": results, "max_rate_meeting_slo": best,
       "gpu_usd_per_1k_requests": (GPU_USD_PER_HR / 3600) / gpu_goodput * 1000 if gpu_goodput else None,
       "assumed_prices_usd_per_hr": {"gpu": GPU_USD_PER_HR, "cpu": CPU_USD_PER_HR},
       "cpu_reference": "see results/RESUME_METRICS.md (4 vCPU baseline goodput 0.93 req/s at SLO)",
       "profiler_top_ops": prof_text}
json.dump(out, open("results/gpu_benchmark.json", "w"), indent=1, default=str)
from google.colab import files; files.download("results/gpu_benchmark.json"); files.download("results/hw_profile_t4.json")